In [4]:
import re
import pandas as pd
from collections import Counter

# === configure ===
out_csv = "/home/lq53/mir_repos/BBOP/random_tests/25Sept_calib_intrinsic_update/all_offsets.csv"

# --- 1) Load and normalize ---
df = pd.read_csv(out_csv)

# extract date from base_path (YYYY_MM_DD)
def extract_date(path: str) -> str:
    if not isinstance(path, str):
        return ""
    m = re.search(r"/(\d{4}_\d{2}_\d{2})/", path)
    return m.group(1) if m else ""

if "date" not in df.columns:
    df["date"] = df["base_path"].map(extract_date)

# coerce offsets to numeric (allow blanks)
for cam in range(1, 7):
    for xy in ("X", "Y"):
        col = f"C{cam}_offset{xy}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# parse date to datetime for proper sorting
df["date_dt"] = pd.to_datetime(df["date"], format="%Y_%m_%d", errors="coerce")

# --- 2) Wide -> long: one row = (session, camera) with offsets and pair label ---
rows = []
for cam in range(1, 7):
    xcol, ycol = f"C{cam}_offsetX", f"C{cam}_offsetY"
    sub = df[["base_path", "date", "date_dt", xcol, ycol]].copy()
    sub["camera"] = cam
    sub.rename(columns={xcol: "offX", ycol: "offY"}, inplace=True)
    rows.append(sub)
long = pd.concat(rows, ignore_index=True)

# drop cameras with missing offsets for that session
long = long.dropna(subset=["offX", "offY"]).copy()
long["offX"] = long["offX"].astype(int)
long["offY"] = long["offY"].astype(int)
long["pair"]  = list(zip(long["offX"], long["offY"]))
long["pair_str"] = long["offX"].astype(str) + "x" + long["offY"].astype(str)

# sort chronologically per camera (tie-break by base_path for stable order)
long = long.sort_values(["camera", "date_dt", "base_path"], kind="mergesort").reset_index(drop=True)

# --- 3) Per-camera: change events (when pair changes compared to previous row) ---
long["prev_pair"] = long.groupby("camera")["pair_str"].shift(1)
long["changed"]   = long["pair_str"] != long["prev_pair"]

change_events = (
    long[long["changed"] & long["prev_pair"].notna()]
    .loc[:, ["camera", "date", "base_path", "prev_pair", "pair_str"]]
    .rename(columns={"prev_pair": "from_pair", "pair_str": "to_pair"})
    .reset_index(drop=True)
)

# --- 4) Per-camera: contiguous periods where the pair stayed constant ---
# We identify runs by cumulative sum of change flags within each camera.
long["run_id"] = long.groupby("camera")["changed"].cumsum()

periods = (
    long.groupby(["camera", "pair_str", "run_id"])
        .agg(
            start_date=("date_dt", "min"),
            end_date=("date_dt", "max"),
            n_sessions=("base_path", "nunique"),
            n_rows=("base_path", "size"),
            dates=("date", lambda s: sorted(set(s))),
            examples=("base_path", lambda s: s.head(2).tolist()),
        )
        .reset_index()
        .sort_values(["camera", "start_date"])
)

# Keep columns tidy and readable
periods["start_date"] = periods["start_date"].dt.strftime("%Y_%m_%d")
periods["end_date"]   = periods["end_date"].dt.strftime("%Y_%m_%d")
periods = periods[["camera", "pair_str", "start_date", "end_date", "n_sessions", "n_rows", "dates", "examples"]]

# --- 5) Per-camera: overall usage summary of each pair (first/last seen, counts) ---
pair_summary_cam = (
    long.groupby(["camera", "pair_str"])
        .agg(
            first_seen=("date_dt", "min"),
            last_seen=("date_dt", "max"),
            n_sessions=("base_path", "nunique"),
            n_rows=("base_path", "size"),
            dates=("date", lambda s: sorted(set(s))),
        )
        .reset_index()
        .sort_values(["camera", "first_seen"])
)

pair_summary_cam["first_seen"] = pair_summary_cam["first_seen"].dt.strftime("%Y_%m_%d")
pair_summary_cam["last_seen"]  = pair_summary_cam["last_seen"].dt.strftime("%Y_%m_%d")

# --- 6) Per-date view: for each (date, camera), what pairs occurred? (flag MIXED) ---
per_date_cam = (
    long.groupby(["date", "camera"])["pair_str"]
        .apply(lambda s: s.iloc[0] if len(set(s)) == 1 else f"MIXED({','.join(sorted(set(s)))})")
        .reset_index(name="pair_or_mixed")
        .sort_values(["date", "camera"])
)

# Optional pivot for a quick glance: rows = date, cols = camera
pivot_date_cam = per_date_cam.pivot(index="date", columns="camera", values="pair_or_mixed").sort_index()

# --- 7) Quick overviews to display in Jupyter ---
print("=== Per-camera offset change events (from → to) ===")
display(change_events.head(30))

print("\n=== Per-camera contiguous periods (constant offsets) ===")
display(periods.head(30))

print("\n=== Per-camera pair usage summary (first/last seen, counts) ===")
display(pair_summary_cam.head(30))

print("\n=== Per-date x camera matrix (pair or MIXED if multiple pairs that day) ===")
display(pivot_date_cam.head(40))

# --- 8) (Optional) Save summaries ---
# pair_summary_cam.to_csv("/tmp/offset_pair_summary_per_camera.csv", index=False)
# periods.to_csv("/tmp/offset_periods_per_camera.csv", index=False)
# change_events.to_csv("/tmp/offset_change_events_per_camera.csv", index=False)
# per_date_cam.to_csv("/tmp/offset_per_date_per_camera.csv", index=False)
# pivot_date_cam.to_csv("/tmp/offset_per_date_per_camera_matrix.csv")


=== Per-camera offset change events (from → to) ===


,camera,date,base_path,from_pair,to_pair
0,4,2024_08_08,/data/big_rim/rsync_dcc_sum/24summ/2024_08_08/...,456x204,360x204
1,6,2024_08_08,/data/big_rim/rsync_dcc_sum/24summ/2024_08_08/...,456x204,144x172



=== Per-camera contiguous periods (constant offsets) ===


,camera,pair_str,start_date,end_date,n_sessions,n_rows,dates,examples
0,1,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
1,2,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
2,3,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
4,4,456x204,2024_06_26,2024_07_19,60,60,"[2024_06_26, 2024_06_27, 2024_06_28, 2024_07_0...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
3,4,360x204,2024_08_08,2025_09_19,222,222,"[, 2024_08_08, 2024_08_26, 2024_08_29, 2024_09...",[/data/big_rim/rsync_dcc_sum/24summ/2024_08_08...
5,5,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
7,6,456x204,2024_06_26,2024_07_19,60,60,"[2024_06_26, 2024_06_27, 2024_06_28, 2024_07_0...",[/data/big_rim/rsync_dcc_sum/24summ/2024_06_26...
6,6,144x172,2024_08_08,2025_09_19,222,222,"[, 2024_08_08, 2024_08_26, 2024_08_29, 2024_09...",[/data/big_rim/rsync_dcc_sum/24summ/2024_08_08...



=== Per-camera pair usage summary (first/last seen, counts) ===


,camera,pair_str,first_seen,last_seen,n_sessions,n_rows,dates
0,1,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07..."
1,2,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07..."
2,3,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07..."
4,4,456x204,2024_06_26,2024_07_19,60,60,"[2024_06_26, 2024_06_27, 2024_06_28, 2024_07_0..."
3,4,360x204,2024_08_08,2025_09_19,222,222,"[, 2024_08_08, 2024_08_26, 2024_08_29, 2024_09..."
5,5,456x204,2024_06_26,2025_09_19,282,282,"[, 2024_06_26, 2024_06_27, 2024_06_28, 2024_07..."
7,6,456x204,2024_06_26,2024_07_19,60,60,"[2024_06_26, 2024_06_27, 2024_06_28, 2024_07_0..."
6,6,144x172,2024_08_08,2025_09_19,222,222,"[, 2024_08_08, 2024_08_26, 2024_08_29, 2024_09..."



=== Per-date x camera matrix (pair or MIXED if multiple pairs that day) ===


camera,1,2,3,4,5,6
date,,,,,,
,456x204,456x204,456x204,360x204,456x204,144x172
2024_06_26,456x204,456x204,456x204,456x204,456x204,456x204
2024_06_27,456x204,456x204,456x204,456x204,456x204,456x204
2024_06_28,456x204,456x204,456x204,456x204,456x204,456x204
2024_07_01,456x204,456x204,456x204,456x204,456x204,456x204
2024_07_02,456x204,456x204,456x204,456x204,456x204,456x204
2024_07_03,456x204,456x204,456x204,456x204,456x204,456x204
2024_07_08,456x204,456x204,456x204,456x204,456x204,456x204
2024_07_09,456x204,456x204,456x204,456x204,456x204,456x204


In [ ]:
import re
import pandas as pd
from collections import Counter

# === configure ===
out_csv = "/home/lq53/mir_repos/BBOP/random_tests/25Sept_calib_intrinsic_update/all_offsets_2.csv"

# --- 1) Load and normalize ---
df = pd.read_csv(out_csv)

# extract date from base_path (YYYY_MM_DD)
def extract_date(path: str) -> str:
    if not isinstance(path, str):
        return ""
    m = re.search(r"/(\d{4}_\d{2}_\d{2})/", path)
    return m.group(1) if m else ""

if "date" not in df.columns:
    df["date"] = df["base_path"].map(extract_date)

# coerce offsets to numeric (allow blanks)
for cam in range(1, 7):
    for xy in ("X", "Y"):
        col = f"C{cam}_offset{xy}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

# parse date to datetime for proper sorting
df["date_dt"] = pd.to_datetime(df["date"], format="%Y_%m_%d", errors="coerce")

# --- 2) Wide -> long: one row = (session, camera) with offsets and pair label ---
rows = []
for cam in range(1, 7):
    xcol, ycol = f"C{cam}_offsetX", f"C{cam}_offsetY"
    sub = df[["base_path", "date", "date_dt", xcol, ycol]].copy()
    sub["camera"] = cam
    sub.rename(columns={xcol: "offX", ycol: "offY"}, inplace=True)
    rows.append(sub)
long = pd.concat(rows, ignore_index=True)

# drop cameras with missing offsets for that session
long = long.dropna(subset=["offX", "offY"]).copy()
long["offX"] = long["offX"].astype(int)
long["offY"] = long["offY"].astype(int)
long["pair"]  = list(zip(long["offX"], long["offY"]))
long["pair_str"] = long["offX"].astype(str) + "x" + long["offY"].astype(str)

# sort chronologically per camera (tie-break by base_path for stable order)
long = long.sort_values(["camera", "date_dt", "base_path"], kind="mergesort").reset_index(drop=True)

# --- 3) Per-camera: change events (when pair changes compared to previous row) ---
long["prev_pair"] = long.groupby("camera")["pair_str"].shift(1)
long["changed"]   = long["pair_str"] != long["prev_pair"]

change_events = (
    long[long["changed"] & long["prev_pair"].notna()]
    .loc[:, ["camera", "date", "base_path", "prev_pair", "pair_str"]]
    .rename(columns={"prev_pair": "from_pair", "pair_str": "to_pair"})
    .reset_index(drop=True)
)

# --- 4) Per-camera: contiguous periods where the pair stayed constant ---
# We identify runs by cumulative sum of change flags within each camera.
long["run_id"] = long.groupby("camera")["changed"].cumsum()

periods = (
    long.groupby(["camera", "pair_str", "run_id"])
        .agg(
            start_date=("date_dt", "min"),
            end_date=("date_dt", "max"),
            n_sessions=("base_path", "nunique"),
            n_rows=("base_path", "size"),
            dates=("date", lambda s: sorted(set(s))),
            examples=("base_path", lambda s: s.head(2).tolist()),
        )
        .reset_index()
        .sort_values(["camera", "start_date"])
)

# Keep columns tidy and readable
periods["start_date"] = periods["start_date"].dt.strftime("%Y_%m_%d")
periods["end_date"]   = periods["end_date"].dt.strftime("%Y_%m_%d")
periods = periods[["camera", "pair_str", "start_date", "end_date", "n_sessions", "n_rows", "dates", "examples"]]

# --- 5) Per-camera: overall usage summary of each pair (first/last seen, counts) ---
pair_summary_cam = (
    long.groupby(["camera", "pair_str"])
        .agg(
            first_seen=("date_dt", "min"),
            last_seen=("date_dt", "max"),
            n_sessions=("base_path", "nunique"),
            n_rows=("base_path", "size"),
            dates=("date", lambda s: sorted(set(s))),
        )
        .reset_index()
        .sort_values(["camera", "first_seen"])
)

pair_summary_cam["first_seen"] = pair_summary_cam["first_seen"].dt.strftime("%Y_%m_%d")
pair_summary_cam["last_seen"]  = pair_summary_cam["last_seen"].dt.strftime("%Y_%m_%d")

# --- 6) Per-date view: for each (date, camera), what pairs occurred? (flag MIXED) ---
per_date_cam = (
    long.groupby(["date", "camera"])["pair_str"]
        .apply(lambda s: s.iloc[0] if len(set(s)) == 1 else f"MIXED({','.join(sorted(set(s)))})")
        .reset_index(name="pair_or_mixed")
        .sort_values(["date", "camera"])
)

# Optional pivot for a quick glance: rows = date, cols = camera
pivot_date_cam = per_date_cam.pivot(index="date", columns="camera", values="pair_or_mixed").sort_index()

# --- 7) Quick overviews to display in Jupyter ---
print("=== Per-camera offset change events (from → to) ===")
display(change_events.head(30))

print("\n=== Per-camera contiguous periods (constant offsets) ===")
display(periods.head(30))

print("\n=== Per-camera pair usage summary (first/last seen, counts) ===")
display(pair_summary_cam.head(30))

print("\n=== Per-date x camera matrix (pair or MIXED if multiple pairs that day) ===")
display(pivot_date_cam.head(40))

# --- 8) (Optional) Save summaries ---
# pair_summary_cam.to_csv("/tmp/offset_pair_summary_per_camera.csv", index=False)
# periods.to_csv("/tmp/offset_periods_per_camera.csv", index=False)
# change_events.to_csv("/tmp/offset_change_events_per_camera.csv", index=False)
# per_date_cam.to_csv("/tmp/offset_per_date_per_camera.csv", index=False)
# pivot_date_cam.to_csv("/tmp/offset_per_date_per_camera_matrix.csv")
